# Train Sparse Autoencoders on nanochat (Google Colab T4)

This notebook trains Sparse Autoencoders (SAEs) on Karpathy's pre-trained **nanochat-d32** model (1.88B params) using Google Colab's **free tier T4 GPU**.

## What You'll Do:
1. Auto-download the pre-trained nanochat-d32 checkpoint from HuggingFace
2. Collect activations from a real text dataset (WikiText-103)
3. Train SAEs to discover interpretable features
4. Visualize what the model learned
5. Save results for further analysis

## Before You Start:
1. **Enable T4 GPU**: Runtime -> Change runtime type -> T4 GPU
2. **Estimated Time**: ~6 minutes end-to-end

---

## What Are Sparse Autoencoders (SAEs)?

**The problem:** Neural networks store knowledge as dense, entangled vectors. A single neuron might partially encode "negation", "questions", and "formal tone" all at once — a problem called **superposition**. This makes it nearly impossible to understand *what* a model learned just by looking at its weights.

**The idea:** An SAE is a small neural network that learns to decompose those dense vectors into **sparse, interpretable features**. Think of it like a prism splitting white light into individual colors:

```
Dense activation (2048 dims) → SAE Encoder → Sparse features (8192 dims, only 32 active) → SAE Decoder → Reconstructed activation
```

- The **encoder** expands the representation (2048 → 8192) and keeps only the top-k most active features
- The **decoder** reconstructs the original from just those few active features
- **Sparsity** is the key: by forcing only ~32 of 8192 features to be "on" at once, each feature is pushed to represent one clean concept

**Why this matters:** If the SAE can reconstruct activations well using only sparse features, those features likely correspond to **meaningful concepts** the model actually uses — things like "this text is a question", "this is a number", or "this has negative sentiment". This is the foundation of **mechanistic interpretability**: understanding neural networks by finding the features they compute with.

**What we'll train:** A **TopK SAE** — the encoder computes all 8192 feature scores, then keeps only the top 32. This gives direct control over sparsity (exactly k=32 active features) without needing to tune a sparsity penalty.

---

## 📦 1. Environment Setup

First, let's verify we have a GPU and install dependencies.

In [ ]:
# Check GPU availability
import torch
import subprocess

print("🔍 Checking GPU...")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Found: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f} GB")
    
    if "T4" in gpu_name:
        print("   Perfect! T4 GPU is ideal for this notebook.")
    else:
        print(f"   Note: This notebook is optimized for T4, but {gpu_name} should work too.")
else:
    print("❌ No GPU found!")
    print("   Go to: Runtime → Change runtime type → Select 'T4 GPU'")
    raise RuntimeError("GPU required for SAE training")

print(f"\n🐍 Python: {subprocess.check_output(['python', '--version']).decode().strip()}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"💻 CUDA: {torch.version.cuda}")

In [ ]:
%%bash
# Install Rust (needed for building the tokenizer)
echo "Installing Rust..."
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y 2>/dev/null
source "$HOME/.cargo/env"
rustc --version
echo "Rust installed!"

In [ ]:
%%bash
# Clone the nanochat-SAE repository
echo "📥 Cloning nanochat-SAE repository..."
if [ ! -d "nanochat-SAE" ]; then
    git clone https://github.com/SolshineCode/nanochat-SAE.git
    cd nanochat-SAE
else
    echo "Repository already cloned."
    cd nanochat-SAE
    git pull
fi

In [ ]:
%%bash
cd nanochat-SAE
source "$HOME/.cargo/env"

# Install Python dependencies directly into Colab's system Python (no venv)
echo "Installing dependencies..."
pip install -q datasets tiktoken tokenizers huggingface_hub tqdm matplotlib maturin psutil regex

# Build Rust tokenizer
echo "Building Rust tokenizer..."
cd rustbpe && maturin develop --release 2>&1 | tail -3
cd ..

echo "Setup complete!"

In [ ]:
# Set up paths (no venv needed - using Colab's system Python)
import sys, os
sys.path.insert(0, '/content/nanochat-SAE')
os.chdir('/content/nanochat-SAE')

import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
print("Ready!")

In [ ]:
# Change to the repo directory and import modules
import os
import sys

# Add to path
sys.path.insert(0, '/content/nanochat-SAE')
os.chdir('/content/nanochat-SAE')

# Now import nanochat modules
import torch
import torch.nn.functional as F
from pathlib import Path
import json
import numpy as np
from tqdm.auto import tqdm

print("✅ All imports successful!")

## 2. Download Pre-trained nanochat-d32 from HuggingFace

Auto-downloads Karpathy's pre-trained 1.88B parameter model.

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

# Download the d32 checkpoint from Karpathy's HuggingFace
print("Downloading nanochat-d32 from HuggingFace...")
MODEL_DIR = Path("/content/nanochat-d32")
MODEL_DIR.mkdir(exist_ok=True)

# Download model weights and metadata
for filename in ["model_000650.pt", "meta_000650.json"]:
    path = hf_hub_download(
        repo_id="karpathy/nanochat-d32",
        filename=filename,
        local_dir=MODEL_DIR,
    )
    size_mb = Path(path).stat().st_size / 1e6
    print(f"  {filename}: {size_mb:.1f} MB")

# Download tokenizer files
TOK_DIR = Path("/content/nanochat-tokenizer")
TOK_DIR.mkdir(exist_ok=True)
for filename in ["token_bytes.pt", "tokenizer.pkl"]:
    path = hf_hub_download(
        repo_id="karpathy/nanochat-d32",
        filename=filename,
        local_dir=TOK_DIR,
    )
    print(f"  {filename}: downloaded")

MODEL_PATH = MODEL_DIR / "model_000650.pt"
print(f"\nCheckpoint ready: {MODEL_PATH}")

## 3. Load the Model

In [ ]:
import json, gc
from nanochat.gpt import GPT, GPTConfig

# Load checkpoint with mmap=True to avoid loading 7.2GB into RAM
# (mmap pages data from disk on demand, keeping RAM usage low)
print(f"Loading model from {MODEL_PATH}...")
checkpoint = torch.load(MODEL_PATH, map_location='cpu', mmap=True)

# Read config from metadata file
meta_path = MODEL_DIR / "meta_000650.json"
if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    print(f"  Metadata: {json.dumps({k:v for k,v in meta.items() if k != 'model'}, indent=2)[:500]}")

# nanochat stores model state dict directly or under 'model' key
if isinstance(checkpoint, dict) and 'model' in checkpoint:
    state_dict = checkpoint['model']
    config_dict = checkpoint.get('config', {})
else:
    state_dict = checkpoint
    config_dict = {}

# Infer config from state dict shapes if not in checkpoint
if not config_dict:
    wte_shape = state_dict.get('transformer.wte.weight',
                state_dict.get('wte.weight', None))
    if wte_shape is not None and isinstance(wte_shape, torch.Tensor):
        vocab_size, n_embd = wte_shape.shape
    else:
        vocab_size, n_embd = 65536, 2048

    # Count layers
    n_layer = 0
    for key in state_dict.keys():
        if 'transformer.h.' in key:
            parts = key.split('.')
            for i, p in enumerate(parts):
                if p == 'h' and i + 1 < len(parts) and parts[i+1].isdigit():
                    n_layer = max(n_layer, int(parts[i+1]) + 1)
    if n_layer == 0:
        n_layer = 32

    config_dict = {
        'vocab_size': int(vocab_size),
        'n_embd': int(n_embd),
        'n_layer': n_layer,
        'n_head': int(n_embd) // 128,
        'n_kv_head': int(n_embd) // 128,
        'sequence_len': 2048,
    }

print(f"  Config: {config_dict}")

# Create model
model_config = GPTConfig(**config_dict)
model = GPT(model_config)

# Load weights from mmap'd checkpoint (strict=False: rotary buffers recomputed)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing:
    print(f"  Missing keys: {len(missing)} (rotary embeddings recomputed at init)")
if unexpected:
    print(f"  Unexpected keys: {len(unexpected)}")

# Free the mmap'd checkpoint BEFORE moving model to GPU
del checkpoint, state_dict
gc.collect()

# Move to GPU in bfloat16 — halves VRAM (3.7GB vs 7.5GB)
# Required for rotary embeddings which assert bfloat16
model = model.to(device='cuda', dtype=torch.bfloat16)
model.eval()
gc.collect()

# Verify rotary embeddings are bfloat16 on cuda
assert model.cos.dtype == torch.bfloat16, f"Rotary embeddings wrong dtype: {model.cos.dtype}"
assert model.cos.device.type == 'cuda', f"Rotary embeddings wrong device: {model.cos.device}"

n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel loaded: {n_params/1e9:.2f}B parameters (bfloat16)")
print(f"  Layers: {model_config.n_layer}, Heads: {model_config.n_head}, d_model: {model_config.n_embd}")
print(f"  GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"  System RAM free: {__import__('psutil').virtual_memory().available/1e9:.1f} GB")

In [ ]:
# Quick sanity check - run a forward pass
with torch.no_grad():
    test_tokens = torch.randint(0, model_config.vocab_size, (1, 64), device='cuda')
    logits = model(test_tokens)
    print(f"Forward pass OK! Output shape: {logits.shape}, dtype: {logits.dtype}")
    print(f"Logit range: [{logits.min().item():.2f}, {logits.max().item():.2f}]")
    print(f"Non-zero logits: {(logits != 0).sum().item()} / {logits.numel()}")
    del logits, test_tokens
    torch.cuda.empty_cache()

## 4. Collect Activations from Real Text

### What are "activations" and why do we need them?

When the model processes text, each layer transforms the input through a series of computations. At each layer, every token is represented as a **2048-dimensional vector** — this is the "residual stream" or "activation". These vectors encode everything the model "knows" about that token in context: its meaning, grammar, sentiment, and relationships to other tokens.

**We collect these vectors so the SAE can learn to decompose them.** Specifically:
- We feed real English text (WikiText-103) through the model
- At **layer 16** (the middle of the 32-layer model), we capture the activation vector for every token
- We collect 50,000 of these vectors to form our SAE training dataset

**Why layer 16?** Middle layers tend to encode the most interesting semantic features — early layers handle syntax and token-level patterns, late layers focus on predicting the next token. Layer 16 is a good default starting point for interpretability research.

**Why WikiText-103?** We need a diverse sample of real English text so the SAE sees a broad range of the model's behavior. The specific dataset matters less than having enough variety — Wikipedia articles cover many topics and writing styles.

In [ ]:
from datasets import load_dataset

# Use WikiText-103 (parquet format, works with modern datasets library)
# Good general English text for collecting activations
print("Downloading WikiText-103 from HuggingFace...")
dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:5000]")
print(f"Loaded {len(dataset)} documents")

# Filter out empty lines
dataset = dataset.filter(lambda x: len(x['text'].strip()) > 50)
print(f"After filtering: {len(dataset)} non-empty documents")

# Preview
sample = dataset[0]['text'][:200]
print(f"\nSample text: {sample}...")

In [ ]:
# Tokenize the dataset for activation collection
from nanochat.tokenizer import RustBPETokenizer

# Load the nanochat tokenizer (tiktoken-based, no Rust needed at runtime)
tokenizer = RustBPETokenizer.from_directory(str(TOK_DIR))
print(f"Tokenizer loaded: vocab_size={tokenizer.get_vocab_size()}")

SEQ_LEN = 512  # Sequence length for activation collection

# Tokenize all documents and concatenate into one long token stream
print("Tokenizing dataset...")
all_tokens = []
for doc in tqdm(dataset, desc="Tokenizing"):
    toks = tokenizer.encode(doc['text'])
    all_tokens.extend(toks)

all_tokens = torch.tensor(all_tokens, dtype=torch.long)
print(f"Total tokens: {len(all_tokens):,}")

# Reshape into (num_sequences, SEQ_LEN) batches
num_seqs = len(all_tokens) // SEQ_LEN
token_batches = all_tokens[:num_seqs * SEQ_LEN].reshape(num_seqs, SEQ_LEN)
print(f"Shaped into {num_seqs} sequences of length {SEQ_LEN}")
print(f"Memory: {token_batches.nbytes / 1e6:.1f} MB")

## 5. Train Sparse Autoencoder (T4 Optimized)

### Understanding the hyperparameters

Before we configure the SAE, here's what each setting controls and why we chose these values:

| Parameter | Value | What it controls |
|---|---|---|
| **Layer** | 16 | Which layer's activations to decompose. Layer 16 is the middle of the 32-layer model — a sweet spot for semantic features. |
| **Expansion factor** | 4x | How many SAE features vs. model dimensions (2048 × 4 = 8192 features). Higher = more features to discover, but more memory and more "dead" features. 4x is conservative for T4; research papers use 8-16x. |
| **k (TopK)** | 32 | Exactly how many features are "on" per token. Lower k = sparser, more interpretable features but worse reconstruction. 32 means each token's activation is explained by just 32 of 8192 possible features. |
| **Num activations** | 50K | How many activation vectors to train on. More = better feature discovery but more RAM. 50K fits comfortably in free tier; research uses 1M+. |
| **Batch size** | 512 | Activations per training step. Standard SAE training value. |
| **Epochs** | 3 | Passes through the training data. 3 is fast iteration; 10-20 gives better results. |

**The key tradeoff:** Expansion factor × k controls the balance between interpretability and reconstruction quality. With 8192 features and k=32, each token gets explained by 32 specific features out of a large vocabulary of possibilities — like describing a scene using only 32 words from a dictionary of 8192.

In [ ]:
# T4-Optimized SAE Configuration for nanochat-d32
# d32 has d_model=2048, 32 layers. T4 has 16GB VRAM, ~12.7GB system RAM.
# Conservative settings to fit in memory.

LAYER = 16  # Middle layer of 32-layer model (good default for interpretability)
D_IN = model_config.n_embd  # 2048 for d32
EXPANSION = 4  # 4x expansion -> 8192 SAE features
D_SAE = D_IN * EXPANSION
K = 32  # TopK active features per token

# 50K activations * 2048 * 4 bytes = ~400MB on CPU — fits in free tier RAM
NUM_ACTIVATIONS = 50_000
COLLECT_BATCH_SIZE = 2  # Small batches — model is 1.88B params on T4

# Training hyperparams
TRAIN_BATCH_SIZE = 512
NUM_EPOCHS = 3
LR = 3e-4

hook_point = f"blocks.{LAYER}.hook_resid_post"

print(f"SAE Config for nanochat-d32 (T4-optimized):")
print(f"  Hook point: {hook_point}")
print(f"  d_in={D_IN}, d_sae={D_SAE}, k={K}")
print(f"  Activations to collect: {NUM_ACTIVATIONS:,}")
print(f"  Training: {NUM_EPOCHS} epochs, batch_size={TRAIN_BATCH_SIZE}, lr={LR}")
print(f"  Estimated activation storage: {NUM_ACTIVATIONS * D_IN * 4 / 1e9:.2f} GB")

In [ ]:
# Collect activations from the model using real text data
# We manually hook the residual stream and collect activations in float32 on CPU

print(f"Collecting {NUM_ACTIVATIONS:,} activations from layer {LAYER}...")

collected = []
total_collected = 0

# Register a forward hook on the target layer's output
target_module = model.transformer.h[LAYER]
activations_buffer = []

def hook_fn(module, input, output):
    # output is the residual stream after this block
    # Shape: (batch, seq_len, d_model)
    activations_buffer.append(output.detach().float().cpu())

handle = target_module.register_forward_hook(hook_fn)

model.eval()
with torch.no_grad():
    pbar = tqdm(total=NUM_ACTIVATIONS, desc="Collecting activations")
    
    for i in range(0, len(token_batches), COLLECT_BATCH_SIZE):
        if total_collected >= NUM_ACTIVATIONS:
            break
        
        batch = token_batches[i:i+COLLECT_BATCH_SIZE].to('cuda')
        
        # Forward pass (activations captured by hook)
        _ = model(batch)
        
        # Flatten (batch, seq, d_model) -> (batch*seq, d_model) and store
        for act in activations_buffer:
            flat = act.reshape(-1, D_IN)
            collected.append(flat)
            total_collected += flat.shape[0]
        
        activations_buffer.clear()
        pbar.update(min(total_collected, NUM_ACTIVATIONS) - pbar.n)
        
        # Free GPU memory periodically
        if i % 20 == 0:
            torch.cuda.empty_cache()
    
    pbar.close()

handle.remove()

# Concatenate and trim to exact count
activations = torch.cat(collected, dim=0)[:NUM_ACTIVATIONS]
del collected
torch.cuda.empty_cache()

print(f"\nCollected activations: {activations.shape}")
print(f"  dtype: {activations.dtype}")
print(f"  Memory: {activations.nbytes / 1e9:.2f} GB")
print(f"  Mean: {activations.mean().item():.4f}")
print(f"  Std: {activations.std().item():.4f}")

In [ ]:
# Train the SAE on collected activations
from sae.config import SAEConfig
from sae.models import TopKSAE
import torch.optim as optim

# Build SAE
sae_config = SAEConfig(
    d_in=D_IN,
    hook_point=hook_point,
    expansion_factor=EXPANSION,
    activation='topk',
    k=K,
)
sae = TopKSAE(sae_config).cuda()

sae_params = sum(p.numel() for p in sae.parameters())
print(f"SAE: {sae_params/1e6:.1f}M parameters ({D_IN} -> {D_SAE} -> {D_IN})")

# Training setup
optimizer = optim.Adam(sae.parameters(), lr=LR)
num_samples = activations.shape[0]
steps_per_epoch = num_samples // TRAIN_BATCH_SIZE

print(f"\nTraining for {NUM_EPOCHS} epochs ({steps_per_epoch} steps/epoch)...")

# Normalize activations (helps SAE training stability)
act_mean = activations.mean(dim=0)
act_std = activations.std(dim=0).clamp(min=1e-6)
activations_norm = (activations - act_mean) / act_std

best_loss = float('inf')
loss_history = []

for epoch in range(NUM_EPOCHS):
    sae.train()
    perm = torch.randperm(num_samples)
    epoch_loss = 0.0
    epoch_steps = 0
    
    pbar = tqdm(range(0, num_samples - TRAIN_BATCH_SIZE, TRAIN_BATCH_SIZE),
                desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    for step_start in pbar:
        idx = perm[step_start:step_start + TRAIN_BATCH_SIZE]
        batch = activations_norm[idx].cuda()
        
        # Forward pass (returns reconstruction, features, metrics)
        reconstructed, feature_acts, metrics = sae(batch)
        loss = metrics['total_loss']
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Normalize decoder weights (keeps features interpretable)
        if sae_config.normalize_decoder:
            sae.normalize_decoder_weights()
        
        epoch_loss += loss.item()
        epoch_steps += 1
        loss_history.append(loss.item())
        
        if epoch_steps % 50 == 0:
            pbar.set_postfix({'loss': f'{loss.item():.6f}', 'l0': f'{metrics["l0"].item():.0f}'})
    
    avg_loss = epoch_loss / max(epoch_steps, 1)
    if avg_loss < best_loss:
        best_loss = avg_loss
    print(f"  Epoch {epoch+1} avg loss: {avg_loss:.6f} (best: {best_loss:.6f})")

# Save SAE + normalization stats
output_dir = Path('/content/sae_outputs') / f"layer_{LAYER}"
output_dir.mkdir(parents=True, exist_ok=True)

torch.save({
    'sae_state_dict': sae.state_dict(),
    'config': sae_config.to_dict(),
    'act_mean': act_mean,
    'act_std': act_std,
    'loss_history': loss_history,
    'best_loss': best_loss,
}, output_dir / 'sae_final.pt')

print(f"\nTraining complete! Best loss: {best_loss:.6f}")
print(f"Saved to {output_dir / 'sae_final.pt'}")

## 6. Evaluate and Visualize

### How to read the metrics

The evaluation will compute several metrics. Here's what each one means and what "good" looks like:

| Metric | What it measures | Our result | Good target |
|---|---|---|---|
| **Explained Variance** | What fraction of the activation's information the SAE captures | ~57% | 80-95% with more data/epochs |
| **MSE Loss** | Average squared reconstruction error per dimension | ~0.42 | Lower is better; depends on activation scale |
| **L0 (active features)** | Average number of non-zero features per token | 32 | Should match k exactly for TopK |
| **Alive features** | Features that activate at least once in the eval set | ~2200/8192 | Higher is better; dead features are wasted capacity |
| **Dead features** | Features that never activate (wasted) | ~73% | <50% is good; reduce by training longer or using more data |

**Is 57% explained variance good?** It's a solid starting point for 50K activations and 3 epochs on a free T4 GPU. Research-grade SAEs trained with 1M+ activations and 10-20 epochs typically achieve 85-95%. The conservative settings here prioritize speed and accessibility — scaling up (more activations, more epochs, larger expansion) will significantly improve these numbers.

**Is 73% dead features bad?** It means most of the 8192 feature "slots" went unused. This is common with limited training data — the SAE didn't see enough variety to populate all features. With more activations (200K+) and epochs (10+), the dead feature rate drops substantially. TopK SAEs are naturally more resistant to dead features than ReLU SAEs.

## 📊 6. Visualize Learned Features

Let's explore what features the SAE discovered!

In [ ]:
# Evaluate SAE quality
print("Evaluating SAE quality...\n")

sae.eval()
with torch.no_grad():
    # Evaluate on a held-out slice
    eval_acts = activations_norm[-10000:].cuda()
    
    # Forward pass through SAE (returns reconstruction, features, metrics)
    reconstructed, feature_acts, metrics = sae(eval_acts)
    
    # MSE
    mse = metrics['mse_loss']
    
    # L0 sparsity (avg active features per token)
    l0 = metrics['l0']
    
    # Explained variance
    total_var = eval_acts.var()
    residual_var = (eval_acts - reconstructed).var()
    explained_var = 1 - (residual_var / total_var)
    
    # Dead features (never activate in eval set)
    feature_max = feature_acts.abs().max(dim=0)[0]
    dead = (feature_max == 0).sum().item()
    alive = D_SAE - dead
    
    # Feature activation frequency
    feature_freq = (feature_acts != 0).float().mean(dim=0)
    top_feats = feature_freq.topk(20)
    
    print(f"SAE Quality Metrics:")
    print(f"  MSE Loss:           {mse.item():.6f}")
    print(f"  L0 (avg active):    {l0.item():.1f} / {K} target")
    print(f"  Explained Variance: {explained_var.item():.1%}")
    print(f"  Alive features:     {alive}/{D_SAE} ({100*alive/D_SAE:.1f}%)")
    print(f"  Dead features:      {dead}/{D_SAE} ({100*dead/D_SAE:.1f}%)")
    
    print(f"\nTop 20 Most Active Features:")
    for i, (freq, idx) in enumerate(zip(top_feats.values, top_feats.indices)):
        mean_act = feature_acts[:, idx.item()].mean().item()
        print(f"  {i+1:2d}. Feature {idx.item():4d}: {freq.item():.2%} freq, mean_act={mean_act:.4f}")

## What You Trained and What to Do Next

### What the SAE learned

You just trained an SAE that decomposes layer 16's residual stream into ~2200 active features (out of 8192 slots). Each feature is a **direction in activation space** that the model uses when processing text. Some of these features likely correspond to interpretable concepts — but identifying *which* concepts requires further analysis.

### Concrete next steps (in order of difficulty)

**1. Improve this SAE (easiest — just change numbers above and re-run)**
- Increase `NUM_ACTIVATIONS` to 200,000 (uses ~1.6GB RAM) for better feature coverage
- Increase `NUM_EPOCHS` to 10 for lower loss and more alive features
- Try `LAYER = 4` (syntax features) or `LAYER = 28` (output-relevant features) and compare

**2. Identify what features represent (intermediate)**
- Save the SAE, then run it on new text and check which features activate for specific inputs
- Example: feed in "The cat is NOT happy" vs "The cat is happy" — which features differ? Those may be negation or sentiment features
- The repo's `scripts/sae_viz.py` generates per-feature dashboards showing top-activating examples

**3. Steer model behavior (advanced)**
- Use `sae.runtime.InterpretableModel` to clamp a feature to zero during generation — does the model stop expressing that concept?
- Amplify a feature by 2-3x — does the model's output shift in a predictable direction?
- This is how researchers test whether features are *causally* meaningful, not just correlational

**4. Compare across layers (research-grade)**
- Train SAEs on layers 4, 16, and 28. Early layers typically show syntactic patterns (word boundaries, punctuation), middle layers show semantic features (topics, sentiment), and late layers show task-specific features (next-token prediction signals)
- This reveals how the model builds up its representation through the layers

### Scaling guide

| Change | Effect | Cost |
|---|---|---|
| 50K → 200K activations | Fewer dead features, better coverage | +1.2GB RAM, +3 min collection |
| 3 → 10 epochs | Lower loss, higher explained variance | +30 sec training |
| 4x → 8x expansion | 16,384 features instead of 8,192 | +134MB SAE, more dead features without more data |
| Layer 16 → multiple layers | Compare feature types across depth | Multiply collection time |
| Colab Free → Pro (A100) | 40GB VRAM, 83GB RAM — enables full-scale training | $10/month |

---

### Resources

- **[SAE_README.md](https://github.com/SolshineCode/nanochat-SAE)** — Full API docs for all SAE modules
- **[COLAB_GUIDE.md](https://github.com/SolshineCode/nanochat-SAE)** — Detailed technical notes and troubleshooting
- **[Anthropic: Towards Monosemanticity](https://transformer-circuits.pub/2023/monosemantic-features)** — The foundational SAE interpretability paper
- **[OpenAI: Scaling SAEs](https://arxiv.org/abs/2406.04093)** — TopK SAE architecture (what we trained here)
- **[Neuronpedia](https://neuronpedia.org)** — Community platform for sharing and exploring SAE features

In [ ]:
# Final summary
print("=" * 70)
print("TRAINING COMPLETE - nanochat-d32 SAE")
print("=" * 70)
print(f"\nModel:   nanochat-d32 ({sum(p.numel() for p in model.parameters())/1e9:.2f}B params)")
print(f"Layer:   {LAYER} ({hook_point})")
print(f"SAE:     {D_IN} -> {D_SAE} -> {D_IN} (TopK, k={K})")
print(f"Data:    {NUM_ACTIVATIONS:,} activations from WikiText-103")
print(f"\nResults:")
print(f"  Explained Variance: {explained_var.item():.1%}")
print(f"  MSE Loss:           {mse.item():.6f}")
print(f"  Alive Features:     {alive}/{D_SAE}")
print(f"  Best Train Loss:    {best_loss:.6f}")
print(f"\nSaved to: {output_dir}")
for f in sorted(output_dir.iterdir()):
    print(f"  {f.name} ({f.stat().st_size/1e6:.1f} MB)")
print("=" * 70)

## 🎯 Next Steps

Congratulations! You've trained a Sparse Autoencoder on nanochat. Here's what you can do next:

### 1. Analyze Features
- Run `scripts/sae_viz.py` to generate interactive feature dashboards
- Identify interpretable features (negation, sentiment, entities, etc.)
- Find which inputs maximally activate each feature

### 2. Feature Steering
- Use `sae.runtime.InterpretableModel` to steer model behavior
- Amplify or suppress specific features during generation
- Test how features affect model outputs

### 3. Multi-Layer Analysis
- Train SAEs on different layers (early vs. late)
- Compare features across layers
- Discover feature composition and circuits

### 4. Share Your Findings
- Upload to Neuronpedia for community exploration
- Write about your discoveries
- Contribute back to the repo!

### 5. Scale Up
- Try larger models (d26, d30)
- Use Colab Pro for better GPUs (A100, V100)
- Train with more activations and larger expansion factors

---

## 📚 Resources

- [nanochat-SAE GitHub](https://github.com/SolshineCode/nanochat-SAE)
- [nanochat Original](https://github.com/karpathy/nanochat)
- [Anthropic: Towards Monosemanticity](https://transformer-circuits.pub/2023/monosemantic-features)
- [OpenAI: Scaling SAEs](https://openai.com/research/sparse-autoencoders)
- [Neuronpedia](https://neuronpedia.org)

---

**Questions or issues?** Open an issue on GitHub or reach out to the community!

**Found something cool?** Share your discoveries! Tweet @karpathy 🎉

In [ ]:
# Final summary
print("=" * 70)
print("TRAINING COMPLETE - nanochat-d32 SAE")
print("=" * 70)
print(f"\nModel:   nanochat-d32 ({sum(p.numel() for p in model.parameters())/1e9:.2f}B params)")
print(f"Layer:   {LAYER} ({hook_point})")
print(f"SAE:     {D_IN} -> {D_SAE} -> {D_IN} (TopK, k={K})")
print(f"Data:    {NUM_ACTIVATIONS:,} activations from OpenWebText")
print(f"\nResults:")
print(f"  Explained Variance: {explained_var.item():.1%}")
print(f"  MSE Loss:           {mse.item():.6f}")
print(f"  Alive Features:     {alive}/{D_SAE}")
print(f"  Best Train Loss:    {best_loss:.6f}")
print(f"\nSaved to: {output_dir}")
for f in sorted(output_dir.iterdir()):
    print(f"  {f.name} ({f.stat().st_size/1e6:.1f} MB)")
print("=" * 70)